# 🗂️ Bloque 1 — Sesión 1: Naturaleza y tipos de datos

**Módulo:** Depuración, limpieza y Clasificación de Datos (5109)  
**Unidad Didáctica:** UD1 — Análisis exploratorio de datos  
**Duración estimada:** ~4 horas.

---

### Contenidos de esta sesión
- **1.1** Naturaleza de los datos: estructurados, semiestructurados y no estructurados
- **1.2** Tipos de variables: numéricas, categóricas, temporales y booleanas

### Criterio de evaluación asociado
> **(CE a)** Se han descrito técnicas y posibilidades de almacenaje de los datos para componer conjuntos, identificando su naturaleza.

In [1]:
# Instalaciones y librerías necesarias (ejecutar una sola vez)
import pandas as pd
import json
import seaborn as sns

print('✅ Librerías cargadas correctamente')

✅ Librerías cargadas correctamente


---
## 1.1 Naturaleza de los datos

Los datos que usaremos para entrenar modelos de Machine Learning pueden tener tres naturalezas distintas según su organización:

| Tipo | Descripción | Ejemplos |
|------|-------------|----------|
| **Estructurados** | Organizados en filas y columnas. Fácilmente tabulables. | Tablas SQL, CSV, Excel |
| **Semiestructurados** | Tienen alguna organización pero no son tabulares. | JSON, XML, HTML |
| **No estructurados** | Sin organización predefinida. | Texto libre, imágenes, audio, vídeo |

> 💡 En este módulo trabajaremos principalmente con datos **estructurados** y **semiestructurados**, que son los más habituales en proyectos de ML supervisado.

### 📌 Ejemplo 1 — Datos estructurados con pandas

In [2]:
# Los datos estructurados se representan fácilmente como un DataFrame
data = {
    'nombre':       ['Ana', 'Luis', 'María', 'Carlos', 'Elena'],
    'edad':         [23, 35, 28, 42, 31],
    'salario':      [28000.0, 45000.0, 32000.0, 55000.0, 38000.0],
    'departamento': ['IT', 'Ventas', 'IT', 'RRHH', 'Ventas'],
    'activo':       [True, True, False, True, True]
}

df = pd.DataFrame(data)
print('Datos estructurados — DataFrame de pandas:')
print(df)
print(f'\nDimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')

Datos estructurados — DataFrame de pandas:
   nombre  edad  salario departamento  activo
0     Ana    23  28000.0           IT    True
1    Luis    35  45000.0       Ventas    True
2   María    28  32000.0           IT   False
3  Carlos    42  55000.0         RRHH    True
4   Elena    31  38000.0       Ventas    True

Dimensiones: 5 filas × 5 columnas


### 📌 Ejemplo 2 — Datos semiestructurados (JSON)

In [3]:
# JSON: tiene estructura, pero no es directamente tabular
# Fíjate en que cada empleado puede tener un número distinto de habilidades
json_data = '''
{
  "empleados": [
    {"nombre": "Ana",    "edad": 23, "habilidades": ["Python", "SQL"]},
    {"nombre": "Luis",   "edad": 35, "habilidades": ["Excel", "Power BI", "SQL"]},
    {"nombre": "María",  "edad": 28, "habilidades": ["Python", "R"]}
  ]
}
'''

datos = json.loads(json_data)
print('Datos semiestructurados — JSON:')
for emp in datos['empleados']:
    print(f"  {emp['nombre']} ({emp['edad']} años): {', '.join(emp['habilidades'])}")

# Intentamos convertirlo a DataFrame
print('\nConvertido a DataFrame con pd.json_normalize():')
df_json = pd.json_normalize(datos['empleados'])
print(df_json)
print('\n⚠️ La columna "habilidades" contiene listas → estructura irregular')

Datos semiestructurados — JSON:
  Ana (23 años): Python, SQL
  Luis (35 años): Excel, Power BI, SQL
  María (28 años): Python, R

Convertido a DataFrame con pd.json_normalize():
  nombre  edad             habilidades
0    Ana    23           [Python, SQL]
1   Luis    35  [Excel, Power BI, SQL]
2  María    28             [Python, R]

⚠️ La columna "habilidades" contiene listas → estructura irregular


### 💡 ¿Cómo entrenar un modelo con datos semiestructurados?

Los modelos de ML necesitan datos tabulares (filas × columnas de **valores escalares**). Cuando los datos semiestructurados contienen listas de longitud variable —como las habilidades del ejemplo anterior— hay que **aplanarlos** antes de poder entrenar cualquier modelo.

La estrategia más habitual es el **multi-hot encoding**: crear una columna binaria por cada valor posible de la lista.

```
JSON / XML  (semiestructurado)
        ↓   pd.json_normalize() + transformaciones
DataFrame tabular  (estructurado)
        ↓   encoding, escalado...
Modelo de ML
```

### 📌 Ejemplo 2b — Aplanar datos semiestructurados con multi-hot encoding

In [4]:
from sklearn.preprocessing import MultiLabelBinarizer

# Partimos del df_json del Ejemplo 2
mlb = MultiLabelBinarizer()
habilidades_encoded = mlb.fit_transform(df_json['habilidades'])

df_modelo = pd.concat([
    df_json[['nombre', 'edad']],
    pd.DataFrame(habilidades_encoded, columns=mlb.classes_)
], axis=1)

print('DataFrame resultante tras el multi-hot encoding:')
print(df_modelo)
print('\nCada habilidad es ahora una columna binaria (1 = tiene la habilidad, 0 = no la tiene).')
print('→ Este formato ya puede usarse directamente con cualquier modelo de scikit-learn.')

DataFrame resultante tras el multi-hot encoding:
  nombre  edad  Excel  Power BI  Python  R  SQL
0    Ana    23      0         0       1  0    1
1   Luis    35      1         1       0  0    1
2  María    28      0         0       1  1    0

Cada habilidad es ahora una columna binaria (1 = tiene la habilidad, 0 = no la tiene).
→ Este formato ya puede usarse directamente con cualquier modelo de scikit-learn.


### 📌 Ejemplo 3 — Datos no estructurados (texto libre)

In [5]:
# Los datos no estructurados no tienen una organización tabular
registros_log = [
    '[2026-06-01 08:32:11] INFO  Entrenamiento completado. Precisión: 94.3%',
    '[2026-06-01 08:35:02] ERROR No se encontró el archivo de datos: datos_test.csv',
    '[2026-06-01 08:36:50] WARN  El conjunto de datos tiene un 12% de valores nulos',
    '[2026-06-01 08:40:17] INFO  Modelo guardado en /models/v2.pkl'
]

print('Datos no estructurados — registros de log (texto libre):')
for registro in registros_log:
    print(' ', registro)

print('\n→ Para extraer información útil necesitaríamos técnicas como:')
print('   expresiones regulares, NLP, tokenización...')

Datos no estructurados — registros de log (texto libre):
  [2026-06-01 08:32:11] INFO  Entrenamiento completado. Precisión: 94.3%
  [2026-06-01 08:35:02] ERROR No se encontró el archivo de datos: datos_test.csv
  [2026-06-01 08:36:50] WARN  El conjunto de datos tiene un 12% de valores nulos
  [2026-06-01 08:40:17] INFO  Modelo guardado en /models/v2.pkl

→ Para extraer información útil necesitaríamos técnicas como:
   expresiones regulares, NLP, tokenización...


### 📌 Ejemplo 3b — Estructurar texto libre con un modelo de lenguaje (LLM)

Las opiniones que escriben los clientes sobre un producto son **datos no estructurados**: texto libre, cada una con su longitud, su tono y sus giros. Para entrenar un modelo (o simplemente para contarlas) necesitamos convertirlas en algo **estructurado**, por ejemplo una columna categórica `sentimiento` con tres valores posibles: `POSITIVO`, `NEGATIVO` o `NEUTRAL`.

Una forma rápida de hacerlo es pedírselo a un **LLM** a través de una API. Usaremos [OpenRouter](https://openrouter.ai), que da acceso a muchos modelos (algunos gratuitos, marcados con `:free`) con la misma librería `openai` que se usa para ChatGPT.

> 🔑 **Antes de ejecutar:** crea una clave gratuita en <https://openrouter.ai/keys> y guárdala en un archivo llamado **`.env`** en la raíz del repositorio `DLCD`, con esta línea:
>
> ```
> OPENROUTER_API_KEY=sk-or-v1-...
> ```
>
> El archivo `.env` está en `.gitignore`, así que **nunca se sube a GitHub**. Si no existe, el cuaderno te pedirá la clave al ejecutar la celda.
>
> ⚠️ Nunca escribas una clave directamente en el código de un cuaderno: cualquiera que lo vea podría usarla.

In [6]:
# DataFrame con opiniones de clientes sobre un producto (datos no estructurados)
opiniones = pd.DataFrame({
    'cliente': ['C001', 'C002', 'C003', 'C004', 'C005', 'C006', 'C007', 'C008'],
    'opinion': [
        'Llegó en dos días y funciona perfectamente. Muy contento con la compra.',
        'La batería no dura ni medio día. Una decepción total, lo voy a devolver.',
        'Es un auricular normal, cumple con lo que promete y poco más.',
        'Calidad de sonido espectacular, lo recomiendo sin dudarlo.',
        'Tardó tres semanas en llegar y la caja venía abollada.',
        'El producto está bien, pero el servicio de atención al cliente fue pésimo.',
        'Lo uso para ir al gimnasio. Ni fu ni fa.',
        '¡Genial! Justo lo que buscaba y a buen precio.',
    ],
})
opiniones

,cliente,opinion
0,C001,Llegó en dos días y funciona perfectamente. Mu...
1,C002,La batería no dura ni medio día. Una decepción...
2,C003,"Es un auricular normal, cumple con lo que prom..."
3,C004,"Calidad de sonido espectacular, lo recomiendo ..."
4,C005,Tardó tres semanas en llegar y la caja venía a...
5,C006,"El producto está bien, pero el servicio de ate..."
6,C007,Lo uso para ir al gimnasio. Ni fu ni fa.
7,C008,¡Genial! Justo lo que buscaba y a buen precio.


In [7]:
# Configuramos el cliente de OpenRouter (usa la librería openai)
import os
from getpass import getpass

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # lee OPENROUTER_API_KEY del archivo .env, si existe
api_key = os.environ.get('OPENROUTER_API_KEY') or getpass('Pega tu clave de OpenRouter: ')

client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=api_key,
)

# Modelo gratuito de OpenRouter. Puedes cambiarlo por cualquier otro de https://openrouter.ai/models
MODELO = 'nvidia/nemotron-3-ultra-550b-a55b:free'
print('✅ Cliente de OpenRouter configurado')

✅ Cliente de OpenRouter configurado


In [8]:
ETIQUETAS = ['POSITIVO', 'NEGATIVO', 'NEUTRAL']

INSTRUCCIONES = (
    'Eres un clasificador de sentimiento de opiniones de clientes. '
    'Responde ÚNICAMENTE con una de estas palabras: POSITIVO, NEGATIVO o NEUTRAL. '
    'Sin explicaciones ni signos de puntuación.'
)


def clasificar_sentimiento(texto):
    """Pide al LLM que clasifique una opinión y devuelve una de las ETIQUETAS."""
    try:
        respuesta = client.chat.completions.create(
            model=MODELO,
            messages=[
                {'role': 'system', 'content': INSTRUCCIONES},
                {'role': 'user', 'content': texto},
            ],
            temperature=0,  # respuestas lo más deterministas posible
        )
        contenido = (respuesta.choices[0].message.content or '').upper()
    except Exception as e:
        print(f'⚠️ Error con la opinión «{texto[:30]}...»: {e}')
        return 'ERROR'

    # El modelo puede añadir texto de más: nos quedamos con la etiqueta que aparezca
    for etiqueta in ETIQUETAS:
        if etiqueta in contenido:
            return etiqueta
    return 'DESCONOCIDO'


# Aplicamos la función a cada opinión → nueva columna estructurada
opiniones['sentimiento'] = opiniones['opinion'].apply(clasificar_sentimiento)
opiniones

,cliente,opinion,sentimiento
0,C001,Llegó en dos días y funciona perfectamente. Mu...,POSITIVO
1,C002,La batería no dura ni medio día. Una decepción...,NEGATIVO
2,C003,"Es un auricular normal, cumple con lo que prom...",NEUTRAL
3,C004,"Calidad de sonido espectacular, lo recomiendo ...",POSITIVO
4,C005,Tardó tres semanas en llegar y la caja venía a...,NEGATIVO
5,C006,"El producto está bien, pero el servicio de ate...",NEGATIVO
6,C007,Lo uso para ir al gimnasio. Ni fu ni fa.,NEUTRAL
7,C008,¡Genial! Justo lo que buscaba y a buen precio.,POSITIVO


In [9]:
# Ahora el texto libre se ha convertido en una variable categórica que podemos analizar
print(opiniones['sentimiento'].value_counts())

sentimiento
POSITIVO    3
NEGATIVO    3
NEUTRAL     2
Name: count, dtype: int64


> 💡 **Lo que acabamos de hacer:** a partir de una columna de **texto no estructurado** hemos generado una columna **categórica nominal** (`sentimiento`) que ya podemos contar, filtrar, graficar o usar como etiqueta para entrenar un modelo propio.
>
> Pero ojo:
> - El LLM **puede equivocarse**, sobre todo con opiniones mixtas (C006) o ambiguas (C007). Hay que revisar una muestra a mano.
> - Los modelos gratuitos tienen **límites de peticiones por minuto**: para miles de filas habría que agrupar opiniones o usar un modelo de pago.
> - Enviamos los textos a un servicio externo: **nunca** mandes datos personales o confidenciales.

---
### 🔧 Ejercicio 1 — Identifica la naturaleza de los datos

Para cada una de las siguientes fuentes de datos, indica si son **estructurados**, **semiestructurados** o **no estructurados**, y justifica brevemente tu respuesta.

In [10]:
# Completa el diccionario con tus respuestas.
# Valores posibles: 'estructurado', 'semiestructurado', 'no estructurado'

fuentes = [
    'Tabla de una base de datos SQL con historial de ventas',
    'Imagen de una radiografía pulmonar',
    'Archivo CSV con temperaturas horarias de una ciudad',
    'Respuesta de una API REST con datos de un pedido en formato JSON',
    'Grabación de audio de una llamada al servicio de atención al cliente',
    'Archivo XML con configuración de un sistema SCADA',
    'Hoja de cálculo Excel con resultados de sensores IoT'
]

respuestas = {
    fuentes[0]: 'estructurado',  # Escribe tu respuesta
    fuentes[1]: 'no estructurado',
    fuentes[2]: 'estructurado',
    fuentes[3]: 'estructurado',
    fuentes[4]: 'no estructurado',
    fuentes[5]: 'estructurado',
    fuentes[6]: 'estructurado',
}

# Verificación
for fuente, respuesta in respuestas.items():
    print(f'{respuesta:20s} → {fuente}')

estructurado         → Tabla de una base de datos SQL con historial de ventas
no estructurado      → Imagen de una radiografía pulmonar
estructurado         → Archivo CSV con temperaturas horarias de una ciudad
estructurado         → Respuesta de una API REST con datos de un pedido en formato JSON
no estructurado      → Grabación de audio de una llamada al servicio de atención al cliente
estructurado         → Archivo XML con configuración de un sistema SCADA
estructurado         → Hoja de cálculo Excel con resultados de sensores IoT


---
## 1.2 Tipos de variables

Dentro de un conjunto de datos estructurado, cada columna es una **variable**. Conocer el tipo de cada variable es fundamental antes de cualquier análisis o limpieza.

```
Variables
├── Numéricas
│   ├── Continuas   → pueden tomar cualquier valor real    (peso, temperatura, precio)
│   └── Discretas   → valores enteros contables            (número de hijos, unidades vendidas)
├── Categóricas
│   ├── Nominales   → sin orden entre categorías           (color, género)
│   └── Ordinales   → con orden definido entre categorías  (nivel educativo, satisfacción 1-5)
├── Temporales     → fechas y horas                       (fecha_nacimiento, timestamp)
└── Booleanas       → solo dos valores: True/False         (activo, tiene_seguro)
```

> ⚠️ **Importante:** pandas asigna tipos automáticamente (`int64`, `float64`, `object`, `bool`, `datetime64`), pero no siempre coinciden con el tipo estadístico real de la variable. Siempre hay que revisarlos.

### 🧱 Tipos de datos de pandas (`dtypes`)

El **tipo estadístico** de una variable (nominal, ordinal, continua…) lo decide quien analiza los datos. Pandas, en cambio, guarda cada columna con un **tipo de dato** (`dtype`), que indica cómo se almacena en memoria y qué operaciones admite. Todas las celdas de una columna tienen el mismo `dtype`, pero cada columna puede tener uno distinto.

Los `dtypes` vienen de dos familias:

**1. Tipos de NumPy** (los que pandas usa por defecto)

| `dtype` | Qué guarda | Observaciones |
|---|---|---|
| `int8`, `int16`, `int32`, `int64` | Enteros con signo | **No admiten nulos**. Si hay un `NaN`, pandas convierte la columna a `float64` |
| `uint8` … `uint64` | Enteros sin signo | Solo valores ≥ 0 |
| `float16`, `float32`, `float64` | Números decimales | El nulo es `NaN` |
| `bool` | `True` / `False` | **No admite nulos** |
| `datetime64[ns]` | Fechas y horas | El nulo es `NaT` |
| `timedelta64[ns]` | Duraciones (diferencia entre fechas) | |
| `complex128` | Números complejos | Poco habitual en datos tabulares |
| `object` | Cualquier objeto de Python | Es donde acaba **el texto** y las columnas **mezcladas** (números y texto a la vez) |

El número indica los bits que ocupa cada valor: `int8` guarda enteros entre −128 y 127 en 1 byte, e `int64` ocupa 8 bytes.

**2. Tipos propios de pandas** (extensiones)

| `dtype` | Qué guarda | Para qué sirve |
|---|---|---|
| `category` | Categorías | Variables **categóricas**. Ocupa mucha menos memoria que `object` |
| `pd.CategoricalDtype([...], ordered=True)` | Categorías con orden | Variables **ordinales**: permite comparar (`<`, `>`) y ordenar por categoría |
| `Int8` … `Int64`, `UInt8` … `UInt64` | Enteros **con nulos** | Van con **mayúscula**. El nulo es `pd.NA` |
| `Float32`, `Float64` | Decimales con nulo `pd.NA` | |
| `boolean` | Booleano **con nulos** | |
| `string` | Texto | Tipo específico para texto (mejor que `object`) |
| `int64[pyarrow]`, `string[pyarrow]`… | Tipos respaldados por Apache Arrow | Más rápidos y ligeros con conjuntos grandes |

**¿Qué `dtype` elijo para cada tipo de variable?**

| Tipo estadístico | `dtype` recomendado | Si la columna tiene nulos |
|---|---|---|
| Numérica continua | `float64` | `float64` (o `Float64`) |
| Numérica discreta | `int64` (o `int8`/`int16` si los valores son pequeños) | `Int64` |
| Categórica nominal | `category` | `category` (los nulos se mantienen) |
| Categórica ordinal | `pd.CategoricalDtype([...], ordered=True)` | Igual |
| Booleana | `bool` | `boolean` |
| Temporal | `datetime64[ns]` | `datetime64[ns]` (nulo `NaT`) |
| Texto libre | `string` (u `object`) | Igual |

**Cómo se cambia el tipo de una columna**

| Herramienta | Uso |
|---|---|
| `df['col'].astype('category')` | Convierte una columna a cualquiera de los tipos de las tablas anteriores |
| `df.astype({'col1': 'category', 'col2': 'int8'})` | Varias columnas de una vez, con un diccionario |
| `pd.to_numeric(s, errors='coerce')` | Texto → número. Lo que no se pueda convertir pasa a `NaN` |
| `pd.to_datetime(s, errors='coerce', format=...)` | Texto → fecha. Lo que no se pueda convertir pasa a `NaT` |
| `s.map({'sí': True, 'no': False})` | Recodificar valores antes de convertirlos |
| `df.convert_dtypes()` | Pasa todas las columnas a los tipos de pandas que admiten nulos |

> ⚠️ **Tres trampas habituales con `astype()`:**
> - `astype('int64')` **falla** si la columna tiene nulos. Usa `'Int64'` (con mayúscula).
> - `astype('bool')` sobre texto da `True` en **cualquier** cadena no vacía, incluida `'no'`. Recodifica antes con `map()`.
> - `astype()` falla en cuanto encuentra un valor que no puede convertir (por ejemplo, `'15,3'` a `float`). Con datos sucios, usa `pd.to_numeric()` o `pd.to_datetime()` con `errors='coerce'`.

### 📌 Ejemplo 4 — Los `dtypes` de NumPy y de pandas en acción

In [ ]:
# Un DataFrame pequeño con los tipos más habituales
demo = pd.DataFrame({
    'entero':      [1, 2, 3],
    'entero_nulo': [1, None, 3],            # entero con un nulo
    'decimal':     [1.5, 2.0, None],
    'booleano':    [True, False, True],
    'texto':       ['rojo', 'verde', 'rojo'],
    'fecha':       pd.to_datetime(['2026-09-28', '2026-10-01', '2026-10-05']),
})

print('Tipos de NumPy que asigna pandas por defecto:')
print(demo.dtypes)
print('\n→ Fíjate en "entero_nulo": NumPy no admite nulos en enteros y pandas la ha pasado a float64.')

# Tipos de pandas: admiten nulos (pd.NA) sin cambiar de tipo
demo_pandas = demo.astype({
    'entero_nulo': 'Int64',     # entero con nulos (con mayúscula)
    'decimal':     'Float64',
    'texto':       'category',
})
print('\nDespués de convertir a tipos de pandas:')
print(demo_pandas.dtypes)
print()
print(demo_pandas)

In [ ]:
# Trampa 1: un entero con nulos no se puede convertir a int64
serie = pd.Series([1, None, 3])
try:
    serie.astype('int64')
except Exception as e:
    print(f'astype("int64") → {type(e).__name__}: {e}')
print('astype("Int64") →', serie.astype('Int64').tolist())

# Trampa 2: astype('bool') sobre texto convierte en True cualquier cadena no vacía
respuestas = pd.Series(['yes', 'no', 'no'])
print('\nastype("bool")      →', respuestas.astype('bool').tolist(), '  ← ¡"no" es True!')
print('map({yes/no}) →', respuestas.map({'yes': True, 'no': False}).tolist())

# Ventaja de category: mucha menos memoria cuando los valores se repiten
colores = pd.Series(['rojo', 'verde', 'azul'] * 100_000)
print(f'\nMemoria como object:   {colores.memory_usage(deep=True) / 1e6:.1f} MB')
print(f'Memoria como category: {colores.astype("category").memory_usage(deep=True) / 1e6:.1f} MB')

### 📌 Ejemplo 5 — Tipos de variables en el dataset Titanic

In [11]:
# Cargamos el dataset Titanic (incluido en seaborn)
titanic = sns.load_dataset('titanic')

print('Dataset: Titanic')
print(f'Dimensiones: {titanic.shape[0]} filas × {titanic.shape[1]} columnas\n')
titanic.head()

Dataset: Titanic
Dimensiones: 891 filas × 15 columnas



,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [ ]:
# Comparamos el tipo que asigna pandas con el tipo estadístico de cada variable
clasificacion = {
    'survived':    'Booleana (0 = no, 1 = sí)',
    'pclass':      'Categórica ordinal (1.ª > 2.ª > 3.ª clase)',
    'sex':         'Categórica nominal',
    'age':         'Numérica continua',
    'sibsp':       'Numérica discreta (nº hermanos/cónyuge a bordo)',
    'parch':       'Numérica discreta (nº padres/hijos a bordo)',
    'fare':        'Numérica continua (precio del billete)',
    'embarked':    'Categórica nominal (puerto de embarque: S, C, Q)',
    'class':       'Categórica ordinal (First > Second > Third)',
    'who':         'Categórica nominal (man/woman/child)',
    'adult_male':  'Booleana',
    'deck':        'Categórica nominal (cubierta A–G)',
    'embark_town': 'Categórica nominal (ciudad de embarque)',
    'alive':       'Booleana (yes/no)',
    'alone':       'Booleana',
}

diagnostico = pd.DataFrame({
    'dtype pandas': titanic.dtypes.astype(str),
    'nulos': titanic.isna().sum(),
    'tipo estadístico': pd.Series(clasificacion),
})
diagnostico

> 💡 **¿Por qué `class` y `deck` ya son `category` y `sex` es `object`?**
> Pandas **nunca** deduce el tipo `category` al leer un fichero: cualquier columna de texto la carga como `object`. Las dos columnas categóricas las ha convertido **seaborn** a mano dentro de `sns.load_dataset('titanic')`, para que sus categorías aparezcan en orden lógico (First → Second → Third, A → G) en los gráficos. Aun así, no las ha marcado como ordinales (`ordered=False`), así que todavía no se pueden comparar con `<` o `>`.
>
> Del diagnóstico sacamos qué hay que corregir:
> - `survived` y `alive` son booleanas, pero están como `int64` y como texto (`yes`/`no`).
> - `pclass` y `class` son ordinales, pero no tienen orden.
> - `sex`, `embarked`, `who` y `embark_town` son nominales, pero están como `object`.
> - `sibsp` y `parch` son enteros pequeños: `int8` basta y ocupa 8 veces menos que `int64`.
> - `age` y `fare` ya son `float64`, que es lo correcto. `age` tiene 177 nulos, que en `float64` se guardan como `NaN` sin problema.
> - `adult_male` y `alone` ya son `bool`.

Corregimos las variables **paso a paso**, agrupándolas por tipo:

In [ ]:
titanic_ok = titanic.copy()

# 1. Booleanas
titanic_ok['survived'] = titanic_ok['survived'].astype('bool')                  # 0/1 → False/True
titanic_ok['alive'] = titanic_ok['alive'].map({'yes': True, 'no': False})        # texto → bool (¡no astype!)

# 2. Categóricas ordinales: definimos el orden de menor a mayor
orden_class = pd.CategoricalDtype(['Third', 'Second', 'First'], ordered=True)
orden_pclass = pd.CategoricalDtype([3, 2, 1], ordered=True)                      # 1.ª clase es la "mayor"
titanic_ok['class'] = titanic_ok['class'].astype(orden_class)
titanic_ok['pclass'] = titanic_ok['pclass'].astype(orden_pclass)

# 3. Categóricas nominales
nominales = ['sex', 'embarked', 'who', 'deck', 'embark_town']
titanic_ok[nominales] = titanic_ok[nominales].astype('category')

# 4. Numéricas discretas: enteros pequeños
titanic_ok[['sibsp', 'parch']] = titanic_ok[['sibsp', 'parch']].astype('int8')

# 5. age, fare, adult_male y alone ya tenían el tipo correcto

comparacion = pd.DataFrame({
    'antes': titanic.dtypes.astype(str),
    'después': titanic_ok.dtypes.astype(str),
    'tipo estadístico': pd.Series(clasificacion),
})
comparacion

In [ ]:
# ¿Qué hemos ganado?

# a) Las ordinales ya se pueden comparar y se ordenan por categoría, no alfabéticamente
print('Pasajeros de clase superior a Third:', (titanic_ok['class'] > 'Third').sum())
print('\nPasajeros por clase (en su orden):')
print(titanic_ok['class'].value_counts(sort=False))

# b) Las booleanas se pueden sumar directamente (True cuenta como 1)
print(f"\nSupervivientes: {titanic_ok['survived'].sum()} de {len(titanic_ok)}")

# c) Menos memoria
antes = titanic.memory_usage(deep=True).sum() / 1024
despues = titanic_ok.memory_usage(deep=True).sum() / 1024
print(f'\nMemoria antes: {antes:.0f} KB  →  después: {despues:.0f} KB ({despues / antes:.0%})')

También se puede hacer **todo en una sola instrucción** pasando un diccionario a `astype()`. La única excepción es `alive`: como es texto, hay que recodificarla antes con `map()`.

> 🔎 Observa que el dataset tiene **columnas redundantes**: `alive` repite `survived`, `class` repite `pclass` y `embark_town` repite `embarked`. Detectarlas es parte de la revisión de la calidad del conjunto que veremos en el Bloque 3.

In [ ]:
tipos = {
    'survived':    'bool',
    'pclass':      orden_pclass,
    'sex':         'category',
    'sibsp':       'int8',
    'parch':       'int8',
    'embarked':    'category',
    'class':       orden_class,
    'who':         'category',
    'deck':        'category',
    'embark_town': 'category',
}

titanic_ok2 = titanic.astype(tipos)
titanic_ok2['alive'] = titanic_ok2['alive'].map({'yes': True, 'no': False})

# Comprobamos que el resultado es idéntico al de la corrección paso a paso
print('¿Mismos tipos?', titanic_ok2.dtypes.equals(titanic_ok.dtypes))
print(titanic_ok2.dtypes)

### 📌 Ejemplo 6 — Cuando pandas se equivoca con el tipo

In [19]:
# Creamos un dataset con errores típicos de tipado
datos_erroneos = pd.DataFrame({
    'edad':          ['23', '35', '28', '42'],         # Debería ser int, está como str
    'precio':        [12.5, 8.0, '15,3', 9.99],        # Coma decimal en vez de punto
    'fecha_compra':  ['2026-01-15', '2026-02-03',
                      '2026-03-20', '2026-01-28'],     # Fecha como texto
    'es_premium':    [1, 0, 1, 1],                     # Booleano codificado como int
    'categoria':     [1, 2, 1, 3]                      # Categórica codificada como int
})


print('Tipos detectados por pandas (antes de corregir):')
print(datos_erroneos.dtypes)
print()
print(datos_erroneos)
print('\n⚠️ Todos estos tipos deberían ser revisados y corregidos antes del análisis.')

#corregimos la variable fecha_compra a tipo fecha
datos_erroneos['fecha_compra'] = pd.to_datetime(datos_erroneos['fecha_compra'], format='%Y-%m-%d', errors='coerce')
print('Tipos detectados por pandas (después de corregir):')
print(datos_erroneos.dtypes)
print()
#Podemos imprimir los datos con la fecha en formato dd-mm-yyyy

datos_mostrar = datos_erroneos.copy()
datos_mostrar['fecha_compra'] = datos_mostrar['fecha_compra'].dt.strftime('%d-%m-%Y')

print(datos_mostrar.to_string(index=False))

Tipos detectados por pandas (antes de corregir):
edad            object
precio          object
fecha_compra    object
es_premium       int64
categoria        int64
dtype: object

  edad precio fecha_compra  es_premium  categoria
0   23   12.5   2026-01-15           1          1
1   35    8.0   2026-02-03           0          2
2   28   15,3   2026-03-20           1          1
3   42   9.99   2026-01-28           1          3

⚠️ Todos estos tipos deberían ser revisados y corregidos antes del análisis.
Tipos detectados por pandas (después de corregir):
edad                    object
precio                  object
fecha_compra    datetime64[ns]
es_premium               int64
categoria                int64
dtype: object

edad precio fecha_compra  es_premium  categoria
  23   12.5   15-01-2026           1          1
  35    8.0   03-02-2026           0          2
  28   15,3   20-03-2026           1          1
  42   9.99   28-01-2026           1          3


---
### 🔧 Ejercicio 2 — Clasifica las variables del dataset Iris

Carga el dataset Iris (disponible en seaborn) y completa las tareas:

In [ ]:
# TAREA 1: Carga el dataset 'iris' con seaborn y muestra las primeras 5 filas
iris = sns.load_dataset('iris')

print(iris.head())

# TAREA 2: Muestra las dimensiones del dataset (filas × columnas)
print(f'Dimensiones: ...')

# TAREA 3: Muestra los tipos de datos que asigna pandas
print(iris.dtypes)

# TAREA 4: Para cada columna, indica su tipo estadístico real:
#   sepal_length → ???
#   sepal_width  → ???
#   petal_length → ???
#   petal_width  → ???
#   species      → ???

# TAREA 5: ¿Algún tipo detectado por pandas no coincide con el tipo estadístico?
# Escribe tu respuesta como comentario:
# Respuesta: ...
iris['species'] = iris['species'].astype('category')
print(iris.dtypes)
print(iris.head())


   sepal_length  sepal_width  petal_length  petal_width species
0           5.1          3.5           1.4          0.2  setosa
1           4.9          3.0           1.4          0.2  setosa
2           4.7          3.2           1.3          0.2  setosa
3           4.6          3.1           1.5          0.2  setosa
4           5.0          3.6           1.4          0.2  setosa
Dimensiones: ...
sepal_length    float64
sepal_width     float64
petal_length    float64
petal_width     float64
species          object
dtype: object
sepal_length     float64
sepal_width      float64
petal_length     float64
petal_width      float64
species         category
dtype: object
   sepal_length  sepal_width  petal_length  petal_width species
0           5.1          3.5           1.4          0.2  setosa
1           4.9          3.0           1.4          0.2  setosa
2           4.7          3.2           1.3          0.2  setosa
3           4.6          3.1           1.5          0.2  setosa
4     

---
### 🔧 Ejercicio 3 — Corrige los tipos del dataset con errores

In [15]:
# Partimos del DataFrame 'datos_erroneos' creado en el Ejemplo 6.
# Corrige los tipos de cada columna para que sean los adecuados.
# Pista: usa pd.to_numeric(), pd.to_datetime(), astype()

df_corregido = datos_erroneos.copy()

# 1. Corrige la columna 'edad' → debe ser int
# df_corregido['edad'] = ...
# 2. Corrige la columna 'precio' → debe ser float (ojo con la coma decimal)
# df_corregido['precio'] = ...

# 3. Corrige 'fecha_compra' → debe ser datetime
# df_corregido['fecha_compra'] = ...

# 4. Corrige 'es_premium' → debe ser bool
# df_corregido['es_premium'] = ...

# 5. Corrige 'categoria' → debe ser categórica
# df_corregido['categoria'] = ...

# Verifica el resultado
# print(df_corregido.dtypes)
# print(df_corregido)

---
## 📝 Resumen de la sesión

| Concepto | Clave |
|----------|-------|
| Datos estructurados | Tablas con filas y columnas. Formato nativo de pandas. |
| Datos semiestructurados | JSON, XML. Tienen organización pero no son directamente tabulares. |
| Datos no estructurados | Texto, imágenes, audio. Requieren técnicas especiales para ML. |
| Estructurar texto con un LLM | Un modelo de lenguaje puede convertir texto libre en una variable categórica (p. ej. sentimiento). |
| Tipo estadístico vs tipo pandas | No siempre coinciden. Siempre revisar con `dtypes`. |
| `dtypes` de NumPy | `int64`, `float64`, `bool`, `datetime64`, `object`. Los enteros y los booleanos no admiten nulos. |
| `dtypes` de pandas | `category`, `CategoricalDtype(ordered=True)`, `Int64`, `Float64`, `boolean`, `string`: admiten nulos (`pd.NA`). |
| Corregir tipos | `astype()` (una columna o un diccionario), `map()` para recodificar, `pd.to_numeric()` y `pd.to_datetime()` para datos sucios. |
| Variables numéricas | Continuas (reales) o discretas (enteras contables). |
| Variables categóricas | Nominales (sin orden) u ordinales (con orden). |

### 🔜 Próxima sesión
**Bloque 1 — Sesión 2:** Identificación de entidades, atributos y relaciones entre conjuntos de datos.